# Pretraining at Scale: the LR schedule, the grad-accum trick, and a real-recipe loop

You already know the objective — predict the next token. At trillion-token scale the objective is the *easy* part; the hard part is the **engineering** around it. This notebook makes two of the non-obvious systems mechanics concrete and *provable*, then runs a tiny end-to-end loop with the real recipe.

1. the **warmup + cosine** learning-rate schedule — print it across steps and confirm it hits the peak exactly at the end of warmup and the floor exactly at the end;
2. **gradient-accumulation equivalence** — prove that $K$ micro-batches of size $m$ produce the *same* gradient as one batch of size $mK$ (the "fake a big batch" trick);
3. a tiny **pretraining loop** with the real recipe (AdamW + warmup→cosine + grad clip) and watch the loss drop on a learnable toy corpus;
4. a toy **MFU** (model FLOPs utilization) from the $6ND$ rule.

**By the end** you'll have proven the two subtle mechanics from scratch and watched the real recipe drive loss down — all on CPU, in seconds.

## Setup

Pick the device at runtime (CUDA → MPS → CPU so it runs anywhere) and seed for reproducibility. Everything below runs on CPU so the printed numbers reproduce on every machine — MPS/CUDA reorder float ops and shift only the low-order digits.

> **Step 0 — imports and device.** Report the available accelerator honestly, but run the reproducible trace on CPU. Print `torch.__version__` so the numbers are pinned to a known build.

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("compute device available:", device, "— the trace below runs on CPU for reproducible numbers")
print("torch:", torch.__version__)
torch.manual_seed(0)

compute device available: mps — the trace below runs on CPU for reproducible numbers
torch: 2.12.0


## 1. The warmup + cosine learning-rate schedule

Large runs don't use a constant LR. They **warm up** linearly from 0 (so a full-size step doesn't wreck random weights before Adam's stats settle), then **cosine-decay** down to a small floor (big steps early, fine steps late).

> **Step 1 — build the schedule from scratch.** Two cases: `step < warmup` is a straight line `peak * step / warmup`; otherwise a half-cosine from peak to floor. The one subtlety is the **off-by-one**: the decay fraction is `(step - warmup) / (total - warmup)`, so the cosine clock *restarts at the end of warmup*. We print sample steps and **assert** the endpoints are exactly the peak (at the end of warmup) and the floor (at the end) — the guard against getting the off-by-one wrong.

In [2]:
def lr_at_step(step, peak=3e-4, floor=3e-5, warmup=10, total=100):
    if step < warmup:                                   # linear warmup: 0 -> peak
        return peak * step / warmup
    progress = (step - warmup) / (total - warmup)       # OFF-BY-ONE FIX: clock restarts at warmup
    return floor + 0.5 * (peak - floor) * (1 + math.cos(math.pi * progress))

schedule = [lr_at_step(t) for t in range(101)]
print("LR schedule (warmup=10, total=100, peak=3e-4, floor=3e-5):")
for t in (0, 5, 10, 30, 55, 100):
    print(f"  step {t:>3}: lr = {schedule[t]:.3e}")
assert abs(schedule[10] - 3e-4) < 1e-12        # peak exactly at the end of warmup
assert abs(schedule[100] - 3e-5) < 1e-12       # floor exactly at the end
print("\nendpoints OK: lr(10) == peak, lr(100) == floor")

LR schedule (warmup=10, total=100, peak=3e-4, floor=3e-5):
  step   0: lr = 0.000e+00
  step   5: lr = 1.500e-04
  step  10: lr = 3.000e-04
  step  30: lr = 2.684e-04
  step  55: lr = 1.650e-04
  step 100: lr = 3.000e-05

endpoints OK: lr(10) == peak, lr(100) == floor


## 2. Gradient-accumulation equivalence

You want a *large* batch (a stable gradient) but it won't fit in memory. **Gradient accumulation** fakes it: run $K$ small micro-batches, accumulate their gradients, then take one optimizer step. The claim is that this gives the *same* update as one big batch of size $mK$ — because the gradient of a mean is the mean of gradients (linearity).

> **Step 2 — the big batch.** Build a tiny linear model and compute the gradient of the mean MSE loss over the **full** batch of $m \times K = 32$ examples in one backward pass. We re-seed before constructing the model so the accumulated version (next cell) starts from the *identical* weights — then any gradient difference is the accumulation logic, not the init.

In [3]:
m, K, dim = 4, 8, 16                                    # micro-batch m, accum steps K, feature dim
torch.manual_seed(1)
X = torch.randn(m * K, dim)                             # full batch of m*K examples
y = torch.randn(m * K, 1)

def fresh_linear():
    torch.manual_seed(2)                                # identical init every call
    return nn.Linear(dim, 1)

big = fresh_linear()                                    # ONE big batch of size m*K
F.mse_loss(big(X), y).backward()                        # mean over all 32 examples -> one gradient
big_grad = big.weight.grad.clone()
print("big-batch gradient shape:", tuple(big_grad.shape), " |grad| =", round(big_grad.norm().item(), 4))

big-batch gradient shape: (1, 16)  |grad| = 1.4164


> **Step 3 — the accumulated version, and the proof.** Now process the *same* data as $K=8$ micro-batches of size $m=4$, calling `.backward()` on each so gradients **accumulate** in `.grad` (no `zero_grad` inside the loop). The crucial detail: divide each micro-batch's mean loss by $K$ **before** backward — averaging $K$ micro-batch means equals the single grand mean. We then assert the accumulated gradient matches the big-batch gradient to float tolerance.

In [4]:
acc = fresh_linear()                                    # identical starting weights as `big`
for k in range(K):
    xb, yb = X[k*m:(k+1)*m], y[k*m:(k+1)*m]             # this micro-batch's slice
    (F.mse_loss(acc(xb), yb) / K).backward()            # /K: mean-of-means == grand mean; grads accumulate
acc_grad = acc.weight.grad.clone()

max_diff = (big_grad - acc_grad).abs().max().item()
print(f"grad-accum equivalence: max|big - accumulated| = {max_diff:.2e}")
assert torch.allclose(big_grad, acc_grad, atol=1e-6)    # identical to float noise
print("PROVED: K micro-batches give the same gradient as one batch of size m*K")

grad-accum equivalence: max|big - accumulated| = 5.96e-08
PROVED: K micro-batches give the same gradient as one batch of size m*K


## 3. A tiny pretraining loop with the real recipe

Now put the recipe to work: train a tiny next-token model with **AdamW + warmup→cosine LR + gradient clipping** and watch the loss fall. We use a *learnable* corpus — a repeating cycle `0,1,2,3,0,1,2,3,...` — so there's a genuine next-token pattern to learn and the descent is clean.

> **Step 4 — the model and the corpus.** The smallest model with next-token logits: embed each token, mix with one nonlinear layer, project to one logit per vocab token. The corpus repeats a length-4 cycle so the next token is predictable. Print the logits shape `[batch, seq, vocab]` — that's what the cross-entropy loss consumes.

In [5]:
V, ctx, period = 16, 8, 4
cycle = torch.arange(period)                            # [0, 1, 2, 3]
data = cycle.repeat(256 // period)                     # [0,1,2,3, 0,1,2,3, ...] LEARNABLE corpus

class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V, 32)                  # token id -> 32-dim vector
        self.ff = nn.Linear(32, 32)                     # one mixing layer
        self.head = nn.Linear(32, V)                    # project to one logit per vocab token
    def forward(self, x):
        return self.head(torch.tanh(self.ff(self.emb(x))))

model = TinyGPT()
demo_x = data[:ctx].unsqueeze(0)                        # [1, ctx]
print("corpus[:12] =", data[:12].tolist())
print("logits shape:", tuple(model(demo_x).shape), " -> [batch, seq, vocab]")

corpus[:12] = [0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3]
logits shape: (1, 8, 16)  -> [batch, seq, vocab]


> **Step 5 — train with the real recipe and watch loss drop.** Each step: grab a random window, set the LR from the warmup→cosine schedule, compute next-token cross-entropy (shift by 1), backward, **clip the global grad norm to 1.0** (the seatbelt against loss spikes), then step. Print loss every 60 steps. Because the corpus is learnable, the loss should drop cleanly from ~`log(16)≈2.8` toward a small value — the recipe working, made visible.

In [6]:
torch.manual_seed(0)
model = TinyGPT()                                       # fresh model for a clean curve
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.1)
TOTAL = 300
print(f"{'step':>4} | {'lr':>9} | {'loss':>7}")
print("-" * 28)
for step in range(TOTAL):
    i = torch.randint(0, len(data) - ctx - 1, (1,)).item()
    x = data[i:i+ctx].unsqueeze(0)                      # [1, ctx] inputs
    tgt = data[i+1:i+ctx+1].unsqueeze(0)               # next-token labels (shift by 1)
    for g in opt.param_groups:
        g["lr"] = lr_at_step(step, 3e-4, 3e-5, 20, TOTAL)  # apply the real schedule
    loss = F.cross_entropy(model(x).reshape(-1, V), tgt.reshape(-1))
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # gradient clipping = the seatbelt
    opt.step()
    if step % 60 == 0 or step == TOTAL - 1:
        print(f"{step:>4} | {lr_at_step(step,3e-4,3e-5,20,TOTAL):.3e} | {loss.item():>7.4f}")

step |        lr |    loss
----------------------------
   0 | 0.000e+00 |  2.8396
  60 | 2.866e-04 |  2.1162
 120 | 2.236e-04 |  1.4196
 180 | 1.350e-04 |  1.0068
 240 | 5.945e-05 |  0.8212
 299 | 3.001e-05 |  0.7507


## 4. Toy MFU: the efficiency scoreboard

**MFU** (Model FLOPs Utilization) is the honest efficiency number: the useful FLOPs you achieve divided by the hardware's peak. The useful rate is $6N \times \text{tokens/sec}$ (the $6N$ from forward+backward per token). Real large runs land at 30–50%.

> **Step 6 — compute a toy MFU.** Count the model's parameters $N$, form $6N$ FLOPs/token, multiply by a *pretend* throughput, and divide by a *pretend* peak. The number is tiny here (the toy model barely uses the pretend hardware) — the point is the **formula** you'd apply to a real run, where it lands at 30–50%.

In [7]:
N = sum(p.numel() for p in model.parameters())          # this model's parameter count
toks_per_sec, peak_flops = 5e4, 1e12                    # pretend measurements
mfu = (6 * N * toks_per_sec) / peak_flops                # 6ND-rate / peak
print(f"params N = {N:,}   training FLOPs/token = 6N = {6*N:,}")
print(f"toy MFU = (6N x {toks_per_sec:.0e} tok/s) / {peak_flops:.0e} peak = {mfu:.4%}")
print("(tiny here because the toy model barely uses the pretend hardware; real runs hit 30-50%)")

params N = 2,096   training FLOPs/token = 6N = 12,576
toy MFU = (6N x 5e+04 tok/s) / 1e+12 peak = 0.0629%
(tiny here because the toy model barely uses the pretend hardware; real runs hit 30-50%)


## Try it yourself

Before you run the cell below, **predict**: in the grad-accum proof, what happens to `max|big - accumulated|` if you **delete the `/ K`** — so each micro-batch contributes its *full* mean gradient instead of $1/K$ of it? Does the difference stay ~`1e-8`, grow to roughly `7×` the big-batch gradient's magnitude, or go to exactly 0?

Then run the cell and check.

*Hint:* without the `/K`, you're **summing** $K$ micro-batch means instead of **averaging** them, so the accumulated gradient is $K=8\times$ too large. The difference jumps to roughly $7\times$ the true gradient's size, and an `allclose` assert would fail. That single missing division is one of the most common real-world large-batch bugs.

In [8]:
bad = fresh_linear()                                    # same init as `big`
for k in range(K):
    xb, yb = X[k*m:(k+1)*m], y[k*m:(k+1)*m]
    F.mse_loss(bad(xb), yb).backward()                  # BUG: no /K -> grads sum instead of averaging
bad_grad = bad.weight.grad.clone()

ratio = bad_grad.norm().item() / big_grad.norm().item()
print(f"|buggy accumulated| / |big| = {ratio:.2f}x   (expected ~{K}x: it SUMMED K means)")
print(f"max|big - buggy| = {(big_grad - bad_grad).abs().max().item():.2e}   (huge, not float noise)")
print("-> a missing /K makes the effective gradient K-times too large: a silent run-wrecking bug")

|buggy accumulated| / |big| = 8.00x   (expected ~8x: it SUMMED K means)
max|big - buggy| = 4.90e+00   (huge, not float noise)
-> a missing /K makes the effective gradient K-times too large: a silent run-wrecking bug


## What we saw

- **The warmup+cosine schedule** ramps linearly to the peak at the end of warmup and cosine-decays to the floor at the end — the two endpoint asserts confirm the **off-by-one** (`(step-warmup)/(total-warmup)`, not `step/total`) is handled.
- **Gradient accumulation is equivalent to a big batch** — 8 micro-batches gave the *same* gradient as one batch of 32 to float tolerance (`~6e-08`), because the gradient of a mean is the mean of gradients. The `/K` is what makes it an *average*, not a sum.
- **The real recipe drives loss down** — AdamW + warmup→cosine + grad clip dropped the loss cleanly from ~2.84 toward ~0.75 on a learnable toy corpus.
- **MFU** is achieved-FLOPs / peak; the $6N$/token rule plugs straight in. Real runs hit 30–50% — the rest is lost to communication, pipeline bubbles, and memory movement.
- **The grad-accum bug** (dropping `/K`) makes the gradient $K\times$ too large — a common, silent, run-wrecking mistake.

**What we skipped** (and where to read it): the **data pipeline** (Common Crawl → dedup → quality filter → mixture), the three kinds of **parallelism** (data/tensor/pipeline) and **ZeRO/FSDP** sharding, the **stability** failure modes, and the **6ND** budgeting in full. All of that is in the [Pretraining at Scale concept page](../02-Pretraining-at-Scale.md), with its [curated references](../02-Pretraining-at-Scale.references.md). The compute-optimal *allocation* (Chinchilla) lives on the [Scaling Laws page](../03-Scaling-Laws/03-Scaling-Laws.md).